# Phase 1 Decoder Smoke

Notebook này là smoke notebook cho Phase 1 decoder. Mục tiêu là kiểm tra đường chạy kỹ thuật trước khi chạy full 15-fold Phase 1.

Giới hạn bắt buộc:
- Smoke run không được dùng để claim hiệu quả model.
- Smoke run không được dùng để claim privileged-transfer efficacy.
- Nếu Phase 1 decoder engine chưa được implement trong repo, notebook phải ghi blocker trung thực và dừng trước bước train.
- Notebook chỉ orchestrate CLI/artifact checks; không tự nhúng scientific/model logic riêng trong notebook.


## Cơ sở tài liệu

Notebook này bám theo các ràng buộc trong docs:

- Phase 1 là `phase1_real`: mainline decoding Groups A/B với comparator suite `A2/A2b/A2c/A2d/A3/A4`.
- Primary endpoint là nested leave-one-subject-out binary decoding memory load 4 vs 8 trong maintenance period.
- Test-time chỉ được dùng scalp EEG; iEEG/teacher chỉ được dùng train-time privileged information.
- Outer-test subject không được tham gia fit preprocessing, ICA, PCA, alignment, latent coupling, observability predictor, calibration, threshold tuning hoặc model selection.
- Phase 1 output bắt buộc về sau gồm fold logs, comparator completeness table, calibration package, negative controls, influence report và claim-state summary.
- A2b/A2c/A2d là comparator bắt buộc cho headline claim; thiếu comparator thì không được claim.

Notebook này chạy smoke 1-2 outer folds khi engine có sẵn. Nếu engine chưa có, notebook ghi `phase1_decoder_smoke_blocker.json`.


In [ ]:
# ============================================================
# Cell 1: Mount Drive and define locked source-of-truth paths
# ============================================================
# Mục tiêu:
# - Mount Google Drive.
# - Khai báo source-of-truth Phase 1 readiness đã revised comparator complete.
# - Khai báo output root cho Phase 1 smoke.
# ============================================================

from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

REPO_DIR = Path('/content/eeg-ds004752')
REPO_URL = 'https://github.com/BrianNguyen29/eeg-ds004752.git'
DRIVE_ROOT = Path('/content/drive/MyDrive/eeg-ds004752')
DATASET_ROOT = DRIVE_ROOT / 'data' / 'ds004752'

GATE0_RUN = DRIVE_ROOT / 'artifacts/gate0/20260417T102811097110Z'
GATE25_RUN = DRIVE_ROOT / 'artifacts/prereg/20260418T161442014597Z'
PREREG_BUNDLE = GATE25_RUN / 'prereg_bundle.json'
COMPARATOR_REVISION_REGISTRY = GATE25_RUN / 'phase1_comparator_revision/phase1_comparator_revision_registry.json'
PHASE05_ESTIMATOR_RUN = DRIVE_ROOT / 'artifacts/phase05_estimators/20260419T130315366518Z'

# Revised readiness sau khi khóa A2b/A2c comparator mapping.
PHASE1_READINESS_RUN = DRIVE_ROOT / 'artifacts/phase1_readiness/20260419T154005857077Z'
PHASE1_READINESS_FILE = PHASE1_READINESS_RUN / 'phase1_input_freeze_revision.json'

PHASE1_SMOKE_ROOT = DRIVE_ROOT / 'artifacts/phase1_smoke'
PHASE1_SMOKE_ROOT.mkdir(parents=True, exist_ok=True)

EXPECTED_PREREG_HASH = '87e928ea747099c336a32121bc156655a1a160d666a251c7ac41228efba96af6'

print('Repo:', REPO_DIR)
print('Dataset root:', DATASET_ROOT)
print('Gate 0 source:', GATE0_RUN)
print('Gate 2.5 source:', GATE25_RUN)
print('Prereg bundle:', PREREG_BUNDLE)
print('Comparator revision registry:', COMPARATOR_REVISION_REGISTRY)
print('Phase 0.5 estimator source:', PHASE05_ESTIMATOR_RUN)
print('Phase 1 readiness source:', PHASE1_READINESS_RUN)
print('Phase 1 smoke output root:', PHASE1_SMOKE_ROOT)


In [ ]:
# ============================================================
# Cell 2: Clone or update private repo
# ============================================================
# Mục tiêu:
# - Đảm bảo repo tồn tại ở /content/eeg-ds004752.
# - Pull code mới nhất.
# - Không in GitHub token ra output.
# ============================================================

import base64
import getpass
import os
import subprocess
import sys


def run(cmd, cwd=None, check=True, capture=False, env=None):
    print('\n$', ' '.join(str(x) for x in cmd))
    result = subprocess.run(
        cmd,
        cwd=cwd,
        check=check,
        text=True,
        capture_output=capture,
        env=env,
    )
    if capture:
        if result.stdout:
            print(result.stdout.strip())
        if result.stderr:
            print(result.stderr.strip())
    return result


def git_auth_header():
    token = os.environ.get('GITHUB_TOKEN')
    if not token:
        token = getpass.getpass('Nhập GitHub token cho repo private: ')
    basic = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
    return f'Authorization: Basic {basic}'


if not REPO_DIR.exists():
    extra_header = git_auth_header()
    run(['git', '-c', f'http.extraHeader={extra_header}', 'clone', REPO_URL, str(REPO_DIR)])
else:
    print('Repo đã tồn tại:', REPO_DIR)

os.chdir(REPO_DIR)
print('Current directory:', Path.cwd())

pull = subprocess.run(['git', 'pull'], cwd=REPO_DIR, text=True)
if pull.returncode != 0:
    extra_header = git_auth_header()
    run(['git', '-c', f'http.extraHeader={extra_header}', 'pull'], cwd=REPO_DIR)

run(['git', 'log', '--oneline', '-5'], cwd=REPO_DIR)
run(['git', 'status', '--short'], cwd=REPO_DIR)
run([sys.executable, '--version'], cwd=REPO_DIR)


In [ ]:
# ============================================================
# Cell 3: Install runtime and run tests
# ============================================================
# Mục tiêu:
# - Cài runtime với scientific extras.
# - Chạy unit tests trước smoke.
# ============================================================

runtime_env = os.environ.copy()
runtime_env['INSTALL_SIGNAL_EXTRAS'] = '1'

run(['bash', 'bootstrap/install_runtime.sh'], cwd=REPO_DIR, env=runtime_env)
run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests'], cwd=REPO_DIR)


In [ ]:
# ============================================================
# Cell 4: Utilities
# ============================================================

import hashlib
import json
from datetime import datetime, timezone


def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()


def sha256_json_content(data: dict, excluded_keys=None) -> str:
    excluded_keys = set(excluded_keys or [])
    clean = {k: v for k, v in data.items() if k not in excluded_keys}
    payload = json.dumps(clean, sort_keys=True, ensure_ascii=False, separators=(',', ':')).encode('utf-8')
    return hashlib.sha256(payload).hexdigest()


def read_json(path: Path):
    if not path.exists():
        raise FileNotFoundError(f'Missing JSON file: {path}')
    return json.loads(path.read_text(encoding='utf-8'))


def write_json(path: Path, data):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, indent=2, ensure_ascii=False) + '\n', encoding='utf-8')


def utc_stamp():
    return datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')


def git_output(args):
    return subprocess.check_output(args, cwd=REPO_DIR, text=True).strip()


print('Utilities ready.')


In [ ]:
# ============================================================
# Cell 5: Verify locked source-of-truth chain
# ============================================================
# Mục tiêu:
# - Kiểm tra các file SOT bắt buộc tồn tại.
# - Kiểm tra Phase 1 readiness revised comparator complete.
# - Kiểm tra prereg hash và comparator revision hash.
# ============================================================

required_paths = {
    'dataset_root': DATASET_ROOT,
    'gate0_manifest': GATE0_RUN / 'manifest.json',
    'gate0_cohort_lock': GATE0_RUN / 'cohort_lock.json',
    'prereg_bundle': PREREG_BUNDLE,
    'comparator_revision_registry': COMPARATOR_REVISION_REGISTRY,
    'phase05_estimator_summary': PHASE05_ESTIMATOR_RUN / 'phase05_estimators_summary.json',
    'phase05_teacher_survival': PHASE05_ESTIMATOR_RUN / 'teacher_survival_table.json',
    'phase05_controls_report': PHASE05_ESTIMATOR_RUN / 'controls_report.json',
    'phase1_readiness_file': PHASE1_READINESS_FILE,
}

for label, path in required_paths.items():
    print(label, path, 'exists =', path.exists())
    if not path.exists():
        raise FileNotFoundError(f'Missing required path: {label} -> {path}')

gate0_manifest = read_json(required_paths['gate0_manifest'])
cohort_lock = read_json(required_paths['gate0_cohort_lock'])
prereg_bundle = read_json(PREREG_BUNDLE)
comparator_revision = read_json(COMPARATOR_REVISION_REGISTRY)
phase05_summary = read_json(required_paths['phase05_estimator_summary'])
phase05_controls = read_json(required_paths['phase05_controls_report'])
phase1_readiness = read_json(PHASE1_READINESS_FILE)

assert gate0_manifest['manifest_status'] == 'signal_audit_ready'
assert gate0_manifest['signal_audit']['status'] == 'ok'
assert gate0_manifest['gate0_blockers'] == []
assert prereg_bundle['status'] == 'locked'
assert prereg_bundle['prereg_bundle_hash_sha256'] == EXPECTED_PREREG_HASH
assert comparator_revision['status'] == 'phase1_comparator_revision_locked'
assert comparator_revision['base_prereg_bundle_hash_sha256'] == EXPECTED_PREREG_HASH
assert phase05_summary['status'] == 'phase05_estimators_controls_complete'
assert phase05_summary['phase05_observability_table_ready'] is True
assert phase05_controls['blockers'] == []
assert phase1_readiness['status'] == 'phase1_input_freeze_revised_comparator_complete'
assert phase1_readiness['authorization']['full_phase1_substantive_run_allowed'] is True

print('\n================ Source Chain OK ================')
print('Gate 0:', GATE0_RUN)
print('Prereg:', PREREG_BUNDLE)
print('Comparator revision:', COMPARATOR_REVISION_REGISTRY)
print('Phase 0.5 estimator:', PHASE05_ESTIMATOR_RUN)
print('Phase 1 readiness:', PHASE1_READINESS_RUN)
print('Phase 1 readiness status:', phase1_readiness['status'])


In [ ]:
# ============================================================
# Cell 6: Environment and GPU check
# ============================================================
# Mục tiêu:
# - Ghi lại Python/Git/GPU/runtime.
# - Smoke có thể chạy T4 nếu engine có sẵn; chưa cần A100.
# ============================================================

env_report = {
    'created_utc': utc_stamp(),
    'python': subprocess.check_output([sys.executable, '--version'], text=True).strip(),
    'git_commit': git_output(['git', 'rev-parse', 'HEAD']),
    'git_branch': git_output(['git', 'rev-parse', '--abbrev-ref', 'HEAD']),
    'git_status_short': git_output(['git', 'status', '--short']),
}

gpu = subprocess.run(['bash', '-lc', 'nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || true'], text=True, capture_output=True)
env_report['gpu'] = gpu.stdout.strip() or 'no_gpu_detected_or_nvidia_smi_unavailable'

print('================ Environment ================')
for key, value in env_report.items():
    print(f'{key}: {value}')


In [ ]:
# ============================================================
# Cell 7: Define smoke scope and no-claim policy
# ============================================================
# Mục tiêu:
# - Khóa smoke chỉ chạy 1-2 outer folds.
# - Chọn deterministic outer-test subjects từ cohort_lock.json đã khóa.
# - Hỗ trợ schema thực tế: participant_id + primary_eligible.
# - Ghi rõ output là smoke_non_claim.
# ============================================================

participants = cohort_lock.get('participants') or []
eligible_subjects = []

for item in participants:
    participant_id = item.get('participant_id') or item.get('subject') or item.get('subject_id')
    is_eligible = (
        item.get('primary_eligible') is True
        or item.get('eligible') is True
        or item.get('is_primary_eligible') is True
    )
    if participant_id and is_eligible:
        eligible_subjects.append(participant_id)

if not eligible_subjects:
    # Fallback 1: Gate 1 n_eff participant_ids nếu được hash-link trong readiness/source chain.
    n_eff_path = Path(phase1_readiness.get('source_of_truth', {}).get('gate1', '')) / 'n_eff_statement.json'
    if n_eff_path.exists():
        n_eff = read_json(n_eff_path)
        eligible_subjects = [x for x in n_eff.get('participant_ids', []) if x]

if not eligible_subjects:
    # Fallback 2: lấy subject/session inventory từ Gate 0 manifest nếu cohort_lock schema thay đổi.
    subject_inventory = gate0_manifest.get('subjects', {}).get('by_subject') or gate0_manifest.get('subject_inventory') or {}
    if isinstance(subject_inventory, dict):
        eligible_subjects = sorted(subject_inventory.keys())

if not eligible_subjects:
    print('cohort_lock keys:', sorted(cohort_lock.keys()))
    print('first cohort participants:', json.dumps(participants[:3], indent=2, ensure_ascii=False))
    print('manifest subjects keys:', sorted((gate0_manifest.get('subjects') or {}).keys()))
    raise ValueError('Could not derive eligible subjects from locked artifacts')

eligible_subjects = sorted(set(eligible_subjects))
expected_n = cohort_lock.get('n_primary_eligible') or gate0_manifest.get('participants', {}).get('n_primary_eligible')
if expected_n is not None and len(eligible_subjects) != int(expected_n):
    print('WARNING: derived eligible subject count differs from locked n_primary_eligible')
    print('derived:', len(eligible_subjects), 'locked:', expected_n)

if len(eligible_subjects) < 2:
    raise ValueError(f'Need at least 2 eligible subjects for smoke; got {eligible_subjects}')

SMOKE_OUTER_TEST_SUBJECTS = eligible_subjects[:2]
SMOKE_MAX_OUTER_FOLDS = len(SMOKE_OUTER_TEST_SUBJECTS)

smoke_policy = {
    'status': 'phase1_decoder_smoke_scope_locked',
    'scope': 'smoke_non_claim',
    'outer_test_subjects': SMOKE_OUTER_TEST_SUBJECTS,
    'max_outer_folds': SMOKE_MAX_OUTER_FOLDS,
    'eligible_subject_count': len(eligible_subjects),
    'eligible_subject_source': 'cohort_lock_or_locked_fallback_artifacts',
    'primary_denominator': 'subject',
    'not_allowed': [
        'Do not report smoke metrics as efficacy evidence.',
        'Do not tune thresholds using smoke outer-test results.',
        'Do not expand comparator set or teacher pool from smoke results.',
        'Do not claim privileged-transfer efficacy from smoke output.',
    ],
}

print('================ Smoke Scope ================')
print('Status:', smoke_policy['status'])
print('Scope:', smoke_policy['scope'])
print('Eligible subject count:', smoke_policy['eligible_subject_count'])
print('Outer-test subjects:', smoke_policy['outer_test_subjects'])
print('Max outer folds:', smoke_policy['max_outer_folds'])


In [ ]:
# ============================================================
# Cell 8: Comparator and artifact expectation table
# ============================================================
# Mục tiêu:
# - Khóa expected comparators cho smoke.
# - Khóa expected output artifacts để theo dõi log chuẩn.
# ============================================================

expected_comparators = ['A2', 'A2b', 'A2c', 'A2d_riemannian', 'A3_distillation', 'A4_privileged']
required_revision_comparators = {'A2b', 'A2c'}
available_revision_comparators = set(comparator_revision.get('mapping', {}).keys())
missing_revision = sorted(required_revision_comparators - available_revision_comparators)
assert not missing_revision, missing_revision

expected_artifacts = [
    'phase1_smoke_inputs.json',
    'phase1_smoke_engine_preflight.json',
    'fold_logs/',
    'comparator_completeness_table.json',
    'calibration_smoke_report.json',
    'negative_controls_smoke_report.json',
    'influence_smoke_report.json',
    'phase1_smoke_summary.json',
    'phase1_smoke_report.md',
]

print('================ Smoke Expected Outputs ================')
print('Expected comparators:', expected_comparators)
print('Expected artifacts:')
for item in expected_artifacts:
    print('-', item)


In [ ]:
# ============================================================
# Cell 9: Phase 1 engine availability preflight
# ============================================================
# Mục tiêu:
# - Kiểm tra repo hiện có Phase 1 decoder engine chưa.
# - Nếu chưa có, ghi blocker trung thực và không chạy train.
# - Nếu sau này engine được implement, cell sẽ phát hiện CLI hỗ trợ smoke args.
# ============================================================

phase1_package = REPO_DIR / 'src' / 'phase1'
phase1_engine_files = [
    phase1_package / '__init__.py',
    phase1_package / 'smoke.py',
]

help_result = subprocess.run(
    [sys.executable, '-m', 'src.cli', 'phase1_real', '--help'],
    cwd=REPO_DIR,
    text=True,
    capture_output=True,
)
phase1_help = help_result.stdout + '\n' + help_result.stderr

required_cli_flags = [
    '--readiness-run',
    '--dataset-root',
    '--smoke',
    '--max-outer-folds',
    '--outer-test-subjects',
]

engine_preflight = {
    'status': 'phase1_decoder_engine_available'
    if phase1_package.exists() and all(flag in phase1_help for flag in required_cli_flags)
    else 'blocked_phase1_decoder_engine_not_implemented',
    'phase1_package_exists': phase1_package.exists(),
    'phase1_engine_files': {str(path): path.exists() for path in phase1_engine_files},
    'phase1_cli_help_returncode': help_result.returncode,
    'required_cli_flags': required_cli_flags,
    'missing_cli_flags': [flag for flag in required_cli_flags if flag not in phase1_help],
    'scientific_integrity_limit': 'If blocked, no decoder smoke was run and no model evidence exists.',
}

print('================ Phase 1 Engine Preflight ================')
print('Status:', engine_preflight['status'])
print('Phase1 package exists:', engine_preflight['phase1_package_exists'])
print('Missing CLI flags:', engine_preflight['missing_cli_flags'])
print('Current phase1_real help:')
print(phase1_help[:3000])


In [ ]:
# ============================================================
# Cell 10: Create smoke run directory and write smoke inputs
# ============================================================
# Mục tiêu:
# - Ghi smoke input record dù engine chưa có.
# - Hash-link tất cả source-of-truth dùng cho smoke.
# ============================================================

phase1_smoke_run = PHASE1_SMOKE_ROOT / utc_stamp()
phase1_smoke_run.mkdir(parents=True, exist_ok=False)

smoke_inputs = {
    'status': 'phase1_decoder_smoke_inputs_locked',
    'created_utc': phase1_smoke_run.name,
    'scope': 'smoke_non_claim',
    'source_of_truth': {
        'gate0': str(GATE0_RUN),
        'prereg_bundle': str(PREREG_BUNDLE),
        'prereg_bundle_hash_sha256': EXPECTED_PREREG_HASH,
        'comparator_revision_registry': str(COMPARATOR_REVISION_REGISTRY),
        'comparator_revision_registry_file_sha256': sha256_file(COMPARATOR_REVISION_REGISTRY),
        'phase05_estimator': str(PHASE05_ESTIMATOR_RUN),
        'phase1_readiness': str(PHASE1_READINESS_RUN),
        'phase1_readiness_file': str(PHASE1_READINESS_FILE),
        'phase1_readiness_file_sha256': sha256_file(PHASE1_READINESS_FILE),
    },
    'dataset_root': str(DATASET_ROOT),
    'smoke_policy': smoke_policy,
    'expected_comparators': expected_comparators,
    'expected_artifacts': expected_artifacts,
    'environment': env_report,
    'engine_preflight': engine_preflight,
}

smoke_inputs_path = phase1_smoke_run / 'phase1_smoke_inputs.json'
write_json(smoke_inputs_path, smoke_inputs)
write_json(phase1_smoke_run / 'phase1_smoke_engine_preflight.json', engine_preflight)
(PHASE1_SMOKE_ROOT / 'latest.txt').write_text(str(phase1_smoke_run), encoding='utf-8')

print('================ Smoke Run Initialized ================')
print('Smoke run:', phase1_smoke_run)
print('Inputs:', smoke_inputs_path)
print('Engine preflight status:', engine_preflight['status'])


In [ ]:
# ============================================================
# Cell 11: Run Phase 1 decoder smoke if engine is available
# ============================================================
# Mục tiêu:
# - Nếu CLI Phase 1 engine có sẵn, chạy smoke 1-2 folds.
# - Nếu chưa có, ghi blocker và không giả lập kết quả.
# ============================================================

if engine_preflight['status'] != 'phase1_decoder_engine_available':
    blocker = {
        'status': 'blocked_phase1_decoder_engine_not_implemented',
        'phase1_smoke_run': str(phase1_smoke_run),
        'reason': 'Current repo exposes phase1_real as prereg guard-only; no Phase 1 decoder smoke engine/CLI flags are available.',
        'missing_cli_flags': engine_preflight['missing_cli_flags'],
        'required_next_implementation': [
            'Implement src.phase1 smoke runner or equivalent CLI-backed Phase 1 engine.',
            'Add phase1_real CLI args for readiness-run, dataset-root, smoke, max-outer-folds, outer-test-subjects.',
            'Ensure outputs include fold logs, comparator completeness, calibration smoke report, negative controls smoke report, influence smoke report.',
        ],
        'scientific_integrity_limits': [
            'No decoder was trained.',
            'No fold metrics were computed.',
            'No privileged-transfer efficacy evidence exists.',
        ],
    }
    write_json(phase1_smoke_run / 'phase1_decoder_smoke_blocker.json', blocker)
    print('================ Phase 1 Smoke Blocked ================')
    print('Status:', blocker['status'])
    print('Reason:', blocker['reason'])
    print('Written:', phase1_smoke_run / 'phase1_decoder_smoke_blocker.json')
else:
    cmd = [
        sys.executable, '-m', 'src.cli', 'phase1_real',
        '--profile', 't4_safe',
        '--config', str(PREREG_BUNDLE),
        '--readiness-run', str(PHASE1_READINESS_RUN),
        '--dataset-root', str(DATASET_ROOT),
        '--output-root', str(phase1_smoke_run),
        '--smoke',
        '--max-outer-folds', str(SMOKE_MAX_OUTER_FOLDS),
        '--outer-test-subjects', *SMOKE_OUTER_TEST_SUBJECTS,
    ]
    print('$', ' '.join(cmd))
    result = subprocess.run(cmd, cwd=REPO_DIR, text=True, capture_output=True)
    print('returncode:', result.returncode)
    print('stdout:', result.stdout[-5000:])
    if result.stderr:
        print('stderr:', result.stderr[-5000:])
    smoke_cli_result = {
        'status': 'phase1_decoder_smoke_cli_passed' if result.returncode == 0 else 'phase1_decoder_smoke_cli_failed',
        'command': cmd,
        'returncode': result.returncode,
        'stdout_tail': result.stdout[-5000:],
        'stderr_tail': result.stderr[-5000:],
    }
    write_json(phase1_smoke_run / 'phase1_decoder_smoke_cli_result.json', smoke_cli_result)
    assert result.returncode == 0, smoke_cli_result


In [ ]:
# ============================================================
# Cell 12: Validate smoke artifacts if smoke ran
# ============================================================
# Mục tiêu:
# - Nếu engine blocked, xác nhận blocker tồn tại.
# - Nếu smoke ran, kiểm tra artifact tối thiểu.
# ============================================================

blocker_path = phase1_smoke_run / 'phase1_decoder_smoke_blocker.json'
cli_result_path = phase1_smoke_run / 'phase1_decoder_smoke_cli_result.json'

if blocker_path.exists():
    blocker = read_json(blocker_path)
    validation = {
        'status': 'phase1_smoke_validation_blocked_before_training',
        'blocker': blocker,
        'checked_artifacts': {
            'inputs': smoke_inputs_path.exists(),
            'engine_preflight': (phase1_smoke_run / 'phase1_smoke_engine_preflight.json').exists(),
            'blocker': blocker_path.exists(),
        },
    }
    print('================ Smoke Artifact Validation ================')
    print('Status:', validation['status'])
    print('Blocker reason:', blocker['reason'])
else:
    required_after_run = [
        phase1_smoke_run / 'fold_logs',
        phase1_smoke_run / 'comparator_completeness_table.json',
        phase1_smoke_run / 'calibration_smoke_report.json',
        phase1_smoke_run / 'negative_controls_smoke_report.json',
        phase1_smoke_run / 'influence_smoke_report.json',
        phase1_smoke_run / 'phase1_smoke_summary.json',
    ]
    missing = [str(path) for path in required_after_run if not path.exists()]
    validation = {
        'status': 'phase1_smoke_artifacts_present' if not missing else 'phase1_smoke_artifacts_missing',
        'missing': missing,
    }
    print('================ Smoke Artifact Validation ================')
    print('Status:', validation['status'])
    print('Missing:', missing)
    assert not missing, validation

write_json(phase1_smoke_run / 'phase1_smoke_validation.json', validation)


In [ ]:
# ============================================================
# Cell 13: Render smoke report
# ============================================================
# Mục tiêu:
# - Tạo report markdown chuẩn để theo dõi kết quả.
# - Không claim model efficacy.
# ============================================================

report_lines = [
    '# Phase 1 Decoder Smoke Report',
    '',
    '## Status',
    '',
    f'- Smoke run: `{phase1_smoke_run}`',
    f'- Engine preflight status: `{engine_preflight["status"]}`',
    f'- Validation status: `{validation["status"]}`',
    f'- Scope: `{smoke_policy["scope"]}`',
    '',
    '## Source Of Truth',
    '',
    f'- Phase 1 readiness: `{PHASE1_READINESS_RUN}`',
    f'- Prereg bundle: `{PREREG_BUNDLE}`',
    f'- Comparator revision registry: `{COMPARATOR_REVISION_REGISTRY}`',
    f'- Phase 0.5 estimator: `{PHASE05_ESTIMATOR_RUN}`',
    '',
    '## Smoke Scope',
    '',
    f'- Outer-test subjects: `{SMOKE_OUTER_TEST_SUBJECTS}`',
    f'- Expected comparators: `{expected_comparators}`',
    '',
    '## Scientific Integrity',
    '',
    '- This smoke notebook is non-claim.',
    '- If blocked, no decoder was trained.',
    '- If smoke ran, smoke metrics are engineering diagnostics only.',
    '- No privileged-transfer efficacy claim is allowed from this notebook.',
]

if blocker_path.exists():
    report_lines.extend([
        '',
        '## Blocker',
        '',
        f'- Status: `{blocker["status"]}`',
        f'- Reason: {blocker["reason"]}',
        f'- Missing CLI flags: `{blocker["missing_cli_flags"]}`',
    ])

report_path = phase1_smoke_run / 'phase1_smoke_report.md'
report_path.write_text('\n'.join(report_lines) + '\n', encoding='utf-8')

print('================ Report Preview ================')
print(report_path.read_text(encoding='utf-8')[:5000])
print('\nReport written:', report_path)


In [ ]:
# ============================================================
# Cell 14: Final interpretation
# ============================================================

print('================ Final Interpretation ================')
print('Phase 1 smoke source of truth:', phase1_smoke_run)
print('Engine preflight status:', engine_preflight['status'])
print('Validation status:', validation['status'])

if blocker_path.exists():
    print('BLOCKED: Phase 1 decoder engine is not implemented in the current repo.')
    print('NEXT: Implement CLI-backed Phase 1 smoke runner before any decoder smoke can run.')
else:
    print('OK: Phase 1 decoder smoke ran and required smoke artifacts were found.')
    print('NEXT: Review fold logs and smoke reports before planning full 15-fold run.')

print('NOT OK TO CLAIM: No claim of decoder efficacy or privileged-transfer efficacy is allowed from this notebook.')
